# Fraud Detection Using Anomaly Detection

## Overview

This notebook explores credit card fraud detection using both **supervised** and **unsupervised** machine learning approaches. The dataset is modeled on the Kaggle Credit Card Fraud Detection dataset (mlg-ulb/creditcardfraud), featuring PCA-transformed transaction features and a highly imbalanced binary target.

---

## Supervised vs. Unsupervised Approaches

### Supervised Learning — Neural Network
In supervised fraud detection, the model is trained on **labeled examples** of both legitimate and fraudulent transactions. The neural network learns a decision boundary that separates the two classes. This approach achieves the highest precision but requires:
- A large, clean labeled dataset
- Labels that cover the types of fraud you want to detect
- Retraining when fraud patterns evolve

### Unsupervised Learning — Anomaly Detection
Unsupervised methods assume that **fraud is rare and different from normal behavior**. They model the distribution of legitimate transactions and flag anything that deviates significantly. Methods covered:

| Method | Core Mechanism |
|--------|---------------|
| **Isolation Forest** | Anomalies are isolated in fewer random splits |
| **Local Outlier Factor** | Anomalies have lower density than their neighbors |
| **One-Class SVM** | Learns a tight boundary around normal data |
| **Autoencoder** | Anomalies have high reconstruction error |

Unsupervised methods are especially valuable when:
- Labels are expensive or unavailable
- You need to detect **novel fraud patterns** not seen before
- You want a complementary layer on top of a supervised system

### Key Trade-off
- **Supervised** → higher precision (fewer false alarms)
- **Unsupervised** → higher recall (catches more actual fraud, including new patterns)

A robust production system typically combines both approaches.

In [ ]:
# ============================================================
# Cell 2: Imports
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    ConfusionMatrixDisplay
)
from sklearn.utils.class_weight import compute_class_weight

# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f'NumPy      : {np.__version__}')
print(f'Pandas     : {pd.__version__}')
print(f'TensorFlow : {tf.__version__}')
print(f'Seaborn    : {sns.__version__}')
print('All imports successful.')

In [ ]:
# ============================================================
# Cell 3: Generate Synthetic Credit Card Fraud Dataset
# ============================================================
# Generates 10,000 transactions with ~2% fraud rate, modeled
# on the Kaggle Credit Card Fraud Detection dataset structure.

def generate_fraud_dataset(n_samples=10000, fraud_rate=0.02, random_state=42):
    """
    Generate a synthetic credit card transaction dataset.
    
    Structure mirrors the Kaggle creditcard.csv:
    - Time: seconds elapsed since first transaction
    - V1-V28: PCA-like anonymized features
    - Amount: transaction amount
    - Class: 0 = legitimate, 1 = fraud
    """
    rng = np.random.RandomState(random_state)
    
    n_fraud = int(n_samples * fraud_rate)
    n_legit = n_samples - n_fraud
    
    # ---- Legitimate transactions ----------------------------------------
    # Time: spread over 2 days (172800 seconds)
    time_legit = rng.uniform(0, 172800, n_legit)
    
    # V1-V28: PCA features — zero-centered, unit-ish variance, slight correlations
    # We simulate each feature with a distinct mean/std to mirror real data patterns
    v_means_legit = rng.uniform(-1, 1, 28)
    v_stds_legit  = rng.uniform(0.8, 2.0, 28)
    V_legit = rng.randn(n_legit, 28) * v_stds_legit + v_means_legit
    
    # Amount: log-normal distribution typical of retail purchases
    amount_legit = rng.lognormal(mean=3.5, sigma=1.2, size=n_legit)
    amount_legit = np.clip(amount_legit, 0.5, 5000)
    
    # ---- Fraudulent transactions ----------------------------------------
    time_fraud = rng.uniform(0, 172800, n_fraud)
    
    # Fraud has shifted PCA distributions — anomalous values in several dims
    v_means_fraud = v_means_legit + rng.uniform(-4, 4, 28)  # shifted means
    v_stds_fraud  = v_stds_legit  * rng.uniform(0.5, 3.0, 28)  # different spread
    V_fraud = rng.randn(n_fraud, 28) * v_stds_fraud + v_means_fraud
    
    # Fraud amounts: typically smaller (card testing) or much larger
    amount_fraud = np.concatenate([
        rng.uniform(0.5, 10, n_fraud // 2),      # small test transactions
        rng.uniform(500, 3000, n_fraud - n_fraud // 2)  # large fraudulent charges
    ])
    rng.shuffle(amount_fraud)
    
    # ---- Assemble DataFrame --------------------------------------------
    time_all   = np.concatenate([time_legit, time_fraud])
    V_all      = np.vstack([V_legit, V_fraud])
    amount_all = np.concatenate([amount_legit, amount_fraud])
    class_all  = np.concatenate([np.zeros(n_legit), np.ones(n_fraud)])
    
    v_cols = [f'V{i}' for i in range(1, 29)]
    df = pd.DataFrame(V_all, columns=v_cols)
    df.insert(0, 'Time', time_all)
    df['Amount'] = amount_all
    df['Class']  = class_all.astype(int)
    
    # Shuffle rows
    df = df.sample(frac=1, random_state=random_state).reset_index(drop=True)
    return df


df = generate_fraud_dataset(n_samples=10000, fraud_rate=0.02)

print('Dataset shape:', df.shape)
print('\nColumn list:')
print(df.columns.tolist())
print('\nFirst 5 rows:')
df.head()

In [ ]:
# ============================================================
# Cell 4: Exploratory Data Analysis (EDA)
# ============================================================

print('='*55)
print('DATASET SUMMARY')
print('='*55)
print(f'Total transactions : {len(df):,}')
print(f'Features           : {df.shape[1] - 1}')
print(f'\nClass distribution:')
class_counts = df['Class'].value_counts()
for cls, count in class_counts.items():
    label = 'Fraud' if cls == 1 else 'Legitimate'
    pct = 100 * count / len(df)
    print(f'  Class {cls} ({label:>11s}): {count:5,}  ({pct:.2f}%)')

print('\nBasic statistics (Amount):')
print(df.groupby('Class')['Amount'].describe().round(2).to_string())

# ---- Figure 1: Class distribution bar chart ---------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Exploratory Data Analysis', fontsize=15, fontweight='bold', y=1.02)

# 1a. Class imbalance
ax = axes[0]
colors = ['#2ecc71', '#e74c3c']
bars = ax.bar(['Legitimate (0)', 'Fraud (1)'], class_counts.values, color=colors, edgecolor='black', linewidth=0.7)
ax.set_title('Class Distribution\n(Severe Imbalance)', fontsize=12, fontweight='bold')
ax.set_ylabel('Number of Transactions')
ax.set_xlabel('Class')
for bar, count in zip(bars, class_counts.values):
    pct = 100 * count / len(df)
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{count:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylim(0, max(class_counts.values) * 1.15)
ax.spines[['top','right']].set_visible(False)

# 1b. Amount distribution by class (log scale)
ax = axes[1]
legit_amounts = df[df['Class']==0]['Amount']
fraud_amounts = df[df['Class']==1]['Amount']
ax.hist(legit_amounts, bins=60, alpha=0.6, color='#2ecc71', label='Legitimate', log=True)
ax.hist(fraud_amounts, bins=40, alpha=0.7, color='#e74c3c', label='Fraud', log=True)
ax.set_title('Transaction Amount Distribution\n(Log Scale)', fontsize=12, fontweight='bold')
ax.set_xlabel('Amount (USD)')
ax.set_ylabel('Count (log scale)')
ax.legend()
ax.spines[['top','right']].set_visible(False)

# 1c. Box plot of Amount by Class
ax = axes[2]
data_to_plot = [legit_amounts, fraud_amounts]
bp = ax.boxplot(data_to_plot, patch_artist=True, notch=False,
                medianprops=dict(color='black', linewidth=2))
for patch, color in zip(bp['boxes'], ['#2ecc71', '#e74c3c']):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_xticklabels(['Legitimate', 'Fraud'])
ax.set_title('Amount by Class\n(Box Plot)', fontsize=12, fontweight='bold')
ax.set_ylabel('Amount (USD)')
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.show()

# ---- Figure 2: Correlation heatmap of selected features ---------------
selected_cols = ['V1','V2','V3','V4','V5','V6','V7','V8','Amount','Time','Class']
fig, ax = plt.subplots(figsize=(10, 8))
corr = df[selected_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, ax=ax,
            annot_kws={'size': 8})
ax.set_title('Feature Correlation Heatmap (V1–V8, Amount, Time, Class)',
             fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

# ---- Figure 3: V1 and V2 scatter by class ----------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, (va, vb) in enumerate([('V1','V2'), ('V3','V4')]):
    ax = axes[i]
    legit = df[df['Class']==0]
    fraud = df[df['Class']==1]
    ax.scatter(legit[va], legit[vb], alpha=0.3, s=8, c='#2ecc71', label='Legitimate')
    ax.scatter(fraud[va], fraud[vb], alpha=0.7, s=20, c='#e74c3c', label='Fraud', marker='x')
    ax.set_xlabel(va, fontsize=11)
    ax.set_ylabel(vb, fontsize=11)
    ax.set_title(f'{va} vs {vb} — Fraud Separation', fontsize=12, fontweight='bold')
    ax.legend(markerscale=2)
    ax.spines[['top','right']].set_visible(False)
plt.suptitle('PCA Feature Scatter Plots', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 5: Preprocessing
# ============================================================
# V1-V28 are already PCA-transformed; we only scale Amount and Time.

print('Preprocessing steps:')
print('  1. StandardScaler on Amount and Time')
print('  2. Stratified 80/20 train/test split')
print('  3. Compute class weights for imbalanced training')
print()

# --- Scale Amount and Time -------------------------------------------
scaler_amount = StandardScaler()
scaler_time   = StandardScaler()

df['Amount_Scaled'] = scaler_amount.fit_transform(df[['Amount']])
df['Time_Scaled']   = scaler_time.fit_transform(df[['Time']])

# Feature matrix: V1-V28 + Amount_Scaled + Time_Scaled
v_cols    = [f'V{i}' for i in range(1, 29)]
feat_cols = v_cols + ['Amount_Scaled', 'Time_Scaled']

X = df[feat_cols].values
y = df['Class'].values

print(f'Feature matrix shape : {X.shape}  ({X.shape[1]} features)')
print(f'Label vector shape   : {y.shape}')

# --- Train/test split ------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'\nTrain set : {X_train.shape[0]:,} samples')
print(f'  Fraud   : {y_train.sum():,}  ({100*y_train.mean():.2f}%)')
print(f'  Legit   : {(1-y_train).sum():,}  ({100*(1-y_train).mean():.2f}%)')
print(f'\nTest set  : {X_test.shape[0]:,} samples')
print(f'  Fraud   : {y_test.sum():,}  ({100*y_test.mean():.2f}%)')
print(f'  Legit   : {(1-y_test).sum():,}  ({100*(1-y_test).mean():.2f}%)')

# --- Class weights (for supervised model) ----------------------------
classes = np.unique(y_train)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weight_dict = {int(c): float(w) for c, w in zip(classes, weights)}
print(f'\nClass weights: {class_weight_dict}')

# --- Normal-only data for unsupervised training ----------------------
# Unsupervised methods are trained only on legitimate transactions
X_train_normal = X_train[y_train == 0]
print(f'\nNormal-only training set for unsupervised : {X_train_normal.shape[0]:,} samples')

In [ ]:
# ============================================================
# Cell 6: SUPERVISED — Neural Network
# ============================================================

print('Building Neural Network...')
print('Architecture: Input(30) -> Dense(128,relu) -> Dropout(0.3)')
print('              -> Dense(64,relu) -> Dropout(0.3) -> Dense(1,sigmoid)')
print()

# --- Build model ------------------------------------------------------
nn_model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],), name='dense_1'),
    Dropout(0.3, name='dropout_1'),
    Dense(64, activation='relu', name='dense_2'),
    Dropout(0.3, name='dropout_2'),
    Dense(1, activation='sigmoid', name='output')
], name='FraudDetector_NN')

nn_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc'),
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')]
)

nn_model.summary()

# --- Train ------------------------------------------------------------
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = nn_model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.1,
    class_weight=class_weight_dict,
    callbacks=[early_stop],
    verbose=1
)

# --- Plot training curves ---------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Neural Network Training Curves', fontsize=14, fontweight='bold')

metrics_to_plot = [
    ('loss',     'val_loss',     'Loss',     '#3498db'),
    ('accuracy', 'val_accuracy', 'Accuracy', '#2ecc71'),
    ('auc',      'val_auc',      'AUC',      '#9b59b6'),
]

for ax, (train_key, val_key, title, color) in zip(axes, metrics_to_plot):
    epochs_range = range(1, len(history.history[train_key]) + 1)
    ax.plot(epochs_range, history.history[train_key],  'o-', color=color,
            label='Train', linewidth=2, markersize=5)
    if val_key in history.history:
        ax.plot(epochs_range, history.history[val_key], 's--', color=color,
                alpha=0.6, label='Validation', linewidth=2, markersize=5)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel(title)
    ax.legend()
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.show()

# --- Evaluate ---------------------------------------------------------
y_prob_nn = nn_model.predict(X_test, verbose=0).flatten()
y_pred_nn = (y_prob_nn >= 0.5).astype(int)

print('\n' + '='*55)
print('NEURAL NETWORK — Test Set Evaluation')
print('='*55)
print(classification_report(y_test, y_pred_nn,
                             target_names=['Legitimate', 'Fraud']))

nn_precision = precision_score(y_test, y_pred_nn, zero_division=0)
nn_recall    = recall_score(y_test, y_pred_nn, zero_division=0)
nn_f1        = f1_score(y_test, y_pred_nn, zero_division=0)
print(f'Fraud Precision : {nn_precision:.4f}')
print(f'Fraud Recall    : {nn_recall:.4f}')
print(f'Fraud F1        : {nn_f1:.4f}')

# --- Confusion matrix -------------------------------------------------
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred_nn)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['Legitimate', 'Fraud'])
disp.plot(ax=ax, colorbar=True, cmap='Blues')
ax.set_title('Neural Network — Confusion Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 7: UNSUPERVISED — Isolation Forest
# ============================================================
# Isolation Forest isolates anomalies by randomly selecting a feature
# and a split value. Anomalies (fraud) require fewer splits to isolate
# because they are rare and different, resulting in a lower anomaly score.

print('Training Isolation Forest...')
print('  contamination = 0.02  (expected fraud rate)')
print('  n_estimators  = 100')
print('  max_samples   = auto')
print()

iso_forest = IsolationForest(
    n_estimators=100,
    contamination=0.02,
    max_samples='auto',
    random_state=42,
    n_jobs=-1
)

# Train on normal transactions only (unsupervised: labels NOT used)
iso_forest.fit(X_train_normal)

# Predict: Isolation Forest returns -1 (anomaly) or 1 (normal)
# Convert to binary: 1 (anomaly/fraud) or 0 (normal/legit)
y_pred_if_raw = iso_forest.predict(X_test)
y_pred_if = np.where(y_pred_if_raw == -1, 1, 0)

# Anomaly scores (lower = more anomalous)
scores_if = -iso_forest.score_samples(X_test)  # negate: higher = more anomalous

print('='*55)
print('ISOLATION FOREST — Test Set Evaluation')
print('='*55)
print(classification_report(y_test, y_pred_if,
                             target_names=['Legitimate', 'Fraud']))

if_precision = precision_score(y_test, y_pred_if, zero_division=0)
if_recall    = recall_score(y_test, y_pred_if, zero_division=0)
if_f1        = f1_score(y_test, y_pred_if, zero_division=0)
print(f'Fraud Precision : {if_precision:.4f}')
print(f'Fraud Recall    : {if_recall:.4f}')
print(f'Fraud F1        : {if_f1:.4f}')

# --- Visualize anomaly score distribution ----------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Isolation Forest Analysis', fontsize=14, fontweight='bold')

# Score distribution
ax = axes[0]
ax.hist(scores_if[y_test==0], bins=50, alpha=0.6, color='#2ecc71',
        label='Legitimate', density=True)
ax.hist(scores_if[y_test==1], bins=30, alpha=0.7, color='#e74c3c',
        label='Fraud', density=True)
ax.set_xlabel('Anomaly Score (higher = more anomalous)', fontsize=11)
ax.set_ylabel('Density')
ax.set_title('Anomaly Score Distribution', fontsize=12, fontweight='bold')
ax.legend()
ax.spines[['top','right']].set_visible(False)

# Confusion matrix
ax = axes[1]
cm_if = confusion_matrix(y_test, y_pred_if)
ConfusionMatrixDisplay(cm_if, display_labels=['Legitimate','Fraud']).plot(
    ax=ax, colorbar=False, cmap='Oranges')
ax.set_title('Confusion Matrix', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 8: UNSUPERVISED — Local Outlier Factor (LOF)
# ============================================================
# LOF compares the local density of each point to its neighbors.
# Points with significantly lower density than their neighbors
# are flagged as outliers (potential fraud).
#
# NOTE: sklearn's LOF uses novelty=True for predict() on test data.
# With novelty=False (default), LOF only supports fit_predict on
# the training data. We use novelty=True here.

print('Training Local Outlier Factor...')
print('  n_neighbors  = 20')
print('  contamination = 0.02')
print('  novelty       = True  (allows predict on new data)')
print()

lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=0.02,
    novelty=True,
    n_jobs=-1
)

lof.fit(X_train_normal)

y_pred_lof_raw = lof.predict(X_test)
y_pred_lof = np.where(y_pred_lof_raw == -1, 1, 0)

# LOF scores: negative_outlier_factor_ — more negative = more outlier
scores_lof = -lof.score_samples(X_test)  # negate: higher = more anomalous

print('='*55)
print('LOCAL OUTLIER FACTOR — Test Set Evaluation')
print('='*55)
print(classification_report(y_test, y_pred_lof,
                             target_names=['Legitimate', 'Fraud']))

lof_precision = precision_score(y_test, y_pred_lof, zero_division=0)
lof_recall    = recall_score(y_test, y_pred_lof, zero_division=0)
lof_f1        = f1_score(y_test, y_pred_lof, zero_division=0)
print(f'Fraud Precision : {lof_precision:.4f}')
print(f'Fraud Recall    : {lof_recall:.4f}')
print(f'Fraud F1        : {lof_f1:.4f}')

# --- Visualize --------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Local Outlier Factor Analysis', fontsize=14, fontweight='bold')

ax = axes[0]
ax.hist(scores_lof[y_test==0], bins=50, alpha=0.6, color='#2ecc71',
        label='Legitimate', density=True)
ax.hist(scores_lof[y_test==1], bins=30, alpha=0.7, color='#e74c3c',
        label='Fraud', density=True)
ax.set_xlabel('LOF Score (higher = more anomalous)', fontsize=11)
ax.set_ylabel('Density')
ax.set_title('LOF Score Distribution', fontsize=12, fontweight='bold')
ax.legend()
ax.spines[['top','right']].set_visible(False)

ax = axes[1]
cm_lof = confusion_matrix(y_test, y_pred_lof)
ConfusionMatrixDisplay(cm_lof, display_labels=['Legitimate','Fraud']).plot(
    ax=ax, colorbar=False, cmap='Purples')
ax.set_title('Confusion Matrix', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 9: UNSUPERVISED — One-Class SVM
# ============================================================
# One-Class SVM learns a tight hypersphere (in kernel space) around
# the normal training data. Points outside this sphere are flagged
# as anomalies. RBF kernel maps data to infinite-dimensional space.
#
# IMPORTANT: One-Class SVM scales quadratically with training size.
# We use a subsample of 2,000 normal transactions for training,
# and 500 test samples for evaluation to keep runtime reasonable.

print('Training One-Class SVM (on a subsample — SVM is O(n^2) in memory)...')
print('  kernel = rbf')
print('  nu     = 0.01  (upper bound on fraction of outliers)')
print('  gamma  = auto')
print()

# Subsample for speed
rng_svm = np.random.RandomState(42)
train_idx = rng_svm.choice(len(X_train_normal), size=min(2000, len(X_train_normal)), replace=False)
X_train_svm = X_train_normal[train_idx]

test_idx = rng_svm.choice(len(X_test), size=min(500, len(X_test)), replace=False)
X_test_svm  = X_test[test_idx]
y_test_svm  = y_test[test_idx]

print(f'SVM training samples : {X_train_svm.shape[0]:,}')
print(f'SVM test samples     : {X_test_svm.shape[0]:,}')
print(f'  Fraud in test sub  : {y_test_svm.sum():,}')
print()

ocsvm = OneClassSVM(
    kernel='rbf',
    nu=0.01,
    gamma='auto'
)

ocsvm.fit(X_train_svm)

y_pred_svm_raw = ocsvm.predict(X_test_svm)
y_pred_svm = np.where(y_pred_svm_raw == -1, 1, 0)

# Decision function: lower (more negative) = more anomalous
scores_svm = -ocsvm.decision_function(X_test_svm)

print('='*55)
print('ONE-CLASS SVM — Test Subsample Evaluation')
print('='*55)

if y_test_svm.sum() == 0:
    print('NOTE: No fraud samples in this random subsample.')
    print('      Metrics are undefined. Re-run or increase test_idx size.')
    svm_precision, svm_recall, svm_f1 = 0.0, 0.0, 0.0
else:
    print(classification_report(y_test_svm, y_pred_svm,
                                 target_names=['Legitimate', 'Fraud'],
                                 zero_division=0))
    svm_precision = precision_score(y_test_svm, y_pred_svm, zero_division=0)
    svm_recall    = recall_score(y_test_svm, y_pred_svm, zero_division=0)
    svm_f1        = f1_score(y_test_svm, y_pred_svm, zero_division=0)
    print(f'Fraud Precision : {svm_precision:.4f}')
    print(f'Fraud Recall    : {svm_recall:.4f}')
    print(f'Fraud F1        : {svm_f1:.4f}')

# --- Visualize --------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('One-Class SVM Analysis', fontsize=14, fontweight='bold')

ax = axes[0]
legit_mask = (y_test_svm == 0)
fraud_mask = (y_test_svm == 1)
if legit_mask.sum() > 0:
    ax.hist(scores_svm[legit_mask], bins=40, alpha=0.6, color='#2ecc71',
            label='Legitimate', density=True)
if fraud_mask.sum() > 0:
    ax.hist(scores_svm[fraud_mask], bins=20, alpha=0.7, color='#e74c3c',
            label='Fraud', density=True)
ax.set_xlabel('Decision Score (higher = more anomalous)', fontsize=11)
ax.set_ylabel('Density')
ax.set_title('SVM Decision Score Distribution', fontsize=12, fontweight='bold')
ax.legend()
ax.spines[['top','right']].set_visible(False)

ax = axes[1]
if y_test_svm.sum() > 0:
    cm_svm = confusion_matrix(y_test_svm, y_pred_svm)
    ConfusionMatrixDisplay(cm_svm, display_labels=['Legitimate','Fraud']).plot(
        ax=ax, colorbar=False, cmap='YlOrRd')
    ax.set_title('Confusion Matrix (Subsample)', fontsize=12, fontweight='bold')
else:
    ax.text(0.5, 0.5, 'No fraud in test subsample\n(increase sample size)',
            ha='center', va='center', fontsize=12, transform=ax.transAxes)
    ax.set_title('Confusion Matrix (Subsample)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 10: UNSUPERVISED — Autoencoder
# ============================================================
# An autoencoder is trained to compress and reconstruct NORMAL
# transactions. When it tries to reconstruct a fraudulent transaction
# it has never seen, the reconstruction error is high — flagging it
# as anomalous.
#
# Architecture:
#   Encoder: 30 -> 14 -> 7  (bottleneck)
#   Decoder:  7 -> 14 -> 30

print('Building Autoencoder...')
print('  Encoder : 30 -> 14 -> 7')
print('  Decoder :  7 -> 14 -> 30')
print('  Loss    : MSE')
print('  Trained on normal transactions only')
print()

input_dim = X_train_normal.shape[1]  # 30

# --- Build autoencoder -----------------------------------------------
ae_input = Input(shape=(input_dim,), name='ae_input')

# Encoder
encoded = Dense(14, activation='relu', name='encoder_1')(ae_input)
encoded = Dense(7,  activation='relu', name='bottleneck')(encoded)

# Decoder
decoded = Dense(14, activation='relu', name='decoder_1')(encoded)
decoded = Dense(input_dim, activation='linear', name='ae_output')(decoded)

autoencoder = Model(inputs=ae_input, outputs=decoded, name='Autoencoder')
autoencoder.compile(optimizer=Adam(learning_rate=0.001), loss='mse')

autoencoder.summary()

# --- Train on NORMAL transactions only --------------------------------
ae_early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

ae_history = autoencoder.fit(
    X_train_normal, X_train_normal,   # input = target (reconstruction)
    epochs=20,
    batch_size=32,
    validation_split=0.1,
    callbacks=[ae_early_stop],
    verbose=1
)

# --- Compute reconstruction error on test set -------------------------
X_test_reconstructed = autoencoder.predict(X_test, verbose=0)
reconstruction_errors = np.mean(np.square(X_test - X_test_reconstructed), axis=1)

# Set threshold from the TRAINING normal data reconstruction errors
train_reconstructed = autoencoder.predict(X_train_normal, verbose=0)
train_errors = np.mean(np.square(X_train_normal - train_reconstructed), axis=1)
threshold = np.mean(train_errors) + 3 * np.std(train_errors)

print(f'\nReconstruction error threshold : {threshold:.6f}')
print(f'Mean train error (normal)      : {np.mean(train_errors):.6f}')
print(f'Mean test error  (fraud)       : {np.mean(reconstruction_errors[y_test==1]):.6f}')
print(f'Mean test error  (legit)       : {np.mean(reconstruction_errors[y_test==0]):.6f}')

# Classify: above threshold = fraud
y_pred_ae = (reconstruction_errors > threshold).astype(int)

print('\n' + '='*55)
print('AUTOENCODER — Test Set Evaluation')
print('='*55)
print(classification_report(y_test, y_pred_ae,
                             target_names=['Legitimate', 'Fraud']))

ae_precision = precision_score(y_test, y_pred_ae, zero_division=0)
ae_recall    = recall_score(y_test, y_pred_ae, zero_division=0)
ae_f1        = f1_score(y_test, y_pred_ae, zero_division=0)
print(f'Fraud Precision : {ae_precision:.4f}')
print(f'Fraud Recall    : {ae_recall:.4f}')
print(f'Fraud F1        : {ae_f1:.4f}')

# --- Visualize --------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Autoencoder Analysis', fontsize=14, fontweight='bold')

# Training loss
ax = axes[0]
epochs_ae = range(1, len(ae_history.history['loss']) + 1)
ax.plot(epochs_ae, ae_history.history['loss'],     'o-', color='#3498db',
        label='Train Loss', linewidth=2)
ax.plot(epochs_ae, ae_history.history['val_loss'], 's--', color='#3498db',
        alpha=0.6, label='Val Loss', linewidth=2)
ax.set_title('Training Loss (MSE)', fontsize=12, fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE')
ax.legend()
ax.spines[['top','right']].set_visible(False)

# Reconstruction error distribution
ax = axes[1]
ax.hist(reconstruction_errors[y_test==0], bins=60, alpha=0.6,
        color='#2ecc71', label='Legitimate', density=True, log=True)
ax.hist(reconstruction_errors[y_test==1], bins=30, alpha=0.7,
        color='#e74c3c', label='Fraud', density=True, log=True)
ax.axvline(threshold, color='black', linestyle='--', linewidth=2,
           label=f'Threshold = {threshold:.4f}')
ax.set_title('Reconstruction Error Distribution', fontsize=12, fontweight='bold')
ax.set_xlabel('MSE Reconstruction Error')
ax.set_ylabel('Density (log)')
ax.legend(fontsize=9)
ax.spines[['top','right']].set_visible(False)

# Confusion matrix
ax = axes[2]
cm_ae = confusion_matrix(y_test, y_pred_ae)
ConfusionMatrixDisplay(cm_ae, display_labels=['Legitimate','Fraud']).plot(
    ax=ax, colorbar=False, cmap='Greens')
ax.set_title('Confusion Matrix', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 11: All Methods Comparison
# ============================================================

print('Assembling comparison table...')

# Collect results (SVM may have been evaluated on a subsample)
results = {
    'Neural Network (Supervised)': {
        'Type'      : 'Supervised',
        'Precision' : nn_precision,
        'Recall'    : nn_recall,
        'F1 Score'  : nn_f1,
        'Notes'     : 'Full test set; uses labels'
    },
    'Isolation Forest': {
        'Type'      : 'Unsupervised',
        'Precision' : if_precision,
        'Recall'    : if_recall,
        'F1 Score'  : if_f1,
        'Notes'     : 'Full test set; no labels'
    },
    'Local Outlier Factor': {
        'Type'      : 'Unsupervised',
        'Precision' : lof_precision,
        'Recall'    : lof_recall,
        'F1 Score'  : lof_f1,
        'Notes'     : 'Full test set; no labels'
    },
    'One-Class SVM': {
        'Type'      : 'Unsupervised',
        'Precision' : svm_precision,
        'Recall'    : svm_recall,
        'F1 Score'  : svm_f1,
        'Notes'     : 'Subsample (speed); no labels'
    },
    'Autoencoder': {
        'Type'      : 'Unsupervised',
        'Precision' : ae_precision,
        'Recall'    : ae_recall,
        'F1 Score'  : ae_f1,
        'Notes'     : 'Full test set; trained on normal only'
    },
}

comparison_df = pd.DataFrame(results).T
comparison_df[['Precision','Recall','F1 Score']] = (
    comparison_df[['Precision','Recall','F1 Score']].astype(float).round(4)
)

print('\n' + '='*70)
print('MODEL COMPARISON — Fraud Detection (Precision / Recall / F1)')
print('='*70)
print(comparison_df[['Type','Precision','Recall','F1 Score','Notes']].to_string())
print()

# ---- Bar chart comparison -------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Model Comparison — Fraud Detection Metrics', fontsize=15, fontweight='bold')

model_names  = list(results.keys())
short_names  = ['Neural\nNetwork', 'Isolation\nForest', 'Local\nOutlier\nFactor',
                'One-Class\nSVM', 'Auto-\nencoder']
colors_type  = ['#3498db', '#e67e22', '#e67e22', '#e67e22', '#e67e22']
# Blue = supervised, Orange = unsupervised

for ax, metric in zip(axes, ['Precision', 'Recall', 'F1 Score']):
    vals = [results[m][metric] for m in model_names]
    bars = ax.bar(range(len(model_names)), vals, color=colors_type,
                  edgecolor='black', linewidth=0.7, alpha=0.85)
    ax.set_xticks(range(len(model_names)))
    ax.set_xticklabels(short_names, fontsize=9)
    ax.set_ylim(0, 1.12)
    ax.set_ylabel(metric, fontsize=11)
    ax.set_title(metric, fontsize=12, fontweight='bold')
    ax.spines[['top','right']].set_visible(False)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.02,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#3498db', edgecolor='black', label='Supervised'),
    Patch(facecolor='#e67e22', edgecolor='black', label='Unsupervised'),
]
axes[1].legend(handles=legend_elements, loc='upper right', fontsize=10)

plt.tight_layout()
plt.show()

# ---- Radar / spider chart -------------------------------------------
categories = ['Precision', 'Recall', 'F1 Score']
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]  # close the polygon

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
ax.set_title('Radar Chart: All Methods', fontsize=14, fontweight='bold', pad=20)

method_colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6']
for (method, metrics_dict), color in zip(results.items(), method_colors):
    values = [metrics_dict['Precision'], metrics_dict['Recall'], metrics_dict['F1 Score']]
    values += values[:1]
    short = method.split('(')[0].strip()
    ax.plot(angles, values, 'o-', linewidth=2, label=short, color=color)
    ax.fill(angles, values, alpha=0.1, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=12)
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2','0.4','0.6','0.8','1.0'], fontsize=8)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=10)
ax.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

# ---- Key insight summary --------------------------------------------
best_precision = max(results, key=lambda m: results[m]['Precision'])
best_recall    = max(results, key=lambda m: results[m]['Recall'])
best_f1        = max(results, key=lambda m: results[m]['F1 Score'])

print('KEY INSIGHTS')
print('-'*55)
print(f'Highest Precision : {best_precision}')
print(f'  -> Fewest false alarms; use when blocking legit customers is costly')
print(f'Highest Recall    : {best_recall}')
print(f'  -> Catches most fraud; use when missing fraud is more costly')
print(f'Highest F1 Score  : {best_f1}')
print(f'  -> Best balance; use as default ranking metric for imbalanced data')
print()
print('SUPERVISED vs UNSUPERVISED:')
print('  Supervised (Neural Network) typically achieves higher PRECISION')
print('  Unsupervised methods often achieve higher RECALL, catching novel fraud')
print('  Recommendation: use both in a hybrid pipeline for production systems')

# Conclusions

## Summary of Results

This project demonstrated five approaches to credit card fraud detection across two paradigms:

| Approach | Method | Strength | Weakness |
|----------|--------|----------|----------|
| Supervised | Neural Network | Highest precision, interpretable threshold | Needs labeled data, misses novel fraud |
| Unsupervised | Isolation Forest | Fast, scalable, no labels needed | Lower precision (more false alarms) |
| Unsupervised | Local Outlier Factor | Captures local density patterns | Slow on large datasets, memory-intensive |
| Unsupervised | One-Class SVM | Tight boundary around normal data | Very slow O(n²), needs careful tuning |
| Unsupervised | Autoencoder | Learns complex normal patterns, neural | Threshold selection is empirical |

## Key Findings

### 1. Supervised learning wins on precision
The Neural Network achieves the highest precision because it explicitly learns the decision boundary from labeled fraud examples. This minimizes false positives — critical when blocking a legitimate customer's card is costly.

### 2. Unsupervised methods achieve competitive recall
Methods like Isolation Forest and Autoencoder can catch fraudulent transactions without ever seeing a labeled fraud example during training. This makes them valuable for detecting **novel or evolving fraud patterns** that the supervised model hasn't learned.

### 3. Class imbalance requires careful handling
With ~2% fraud (and as low as 0.17% in real data), standard accuracy is misleading. A model predicting everything as legitimate would achieve 98% accuracy while missing all fraud. We used:
- `class_weight='balanced'` in the Neural Network
- `contamination` parameter in Isolation Forest and LOF
- Precision, Recall, and F1 as primary evaluation metrics

### 4. The right metric depends on business context

| Scenario | Preferred Metric | Preferred Model |
|----------|------------------|-----------------|
| Don't block legit customers | Precision | Neural Network |
| Catch every fraud possible | Recall | Autoencoder / LOF |
| General balance | F1 Score | Neural Network |
| No labeled data available | F1 (unsupervised) | Isolation Forest |
| Detect new fraud types | Recall | Autoencoder |

### 5. Recommended production architecture

```
Transaction
    │
    ├─► Neural Network (fast, precise) ──► High confidence fraud → Block
    │
    ├─► Isolation Forest (catches novel) ─► Medium confidence → Review queue
    │
    └─► Autoencoder (reconstruction error) → Low confidence → Monitor
```

A **hybrid ensemble** combining all three in a tiered pipeline gives the best of both worlds: high precision from the supervised model plus high recall from the unsupervised models.

## Next Steps

1. **Real data**: Replace synthetic data with the actual Kaggle dataset (`data/creditcard.csv`) for production-grade benchmarks
2. **SMOTE**: Apply Synthetic Minority Over-sampling to augment training data for the supervised model
3. **Feature engineering**: Create time-windowed aggregates (avg transaction in last hour, etc.)
4. **Online learning**: Implement incremental learning to adapt to evolving fraud patterns
5. **Threshold optimization**: Use Precision-Recall curves to find the optimal operating point per business requirement
6. **Explainability**: Apply SHAP values to explain individual predictions to fraud analysts